In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Write a GPU program that interleaves two arrays of 32-bit floating point numbers.
  Given two input arrays <code>A</code> and <code>B</code>, each of length <code>N</code>,
  produce an output array of length <code>2N</code> where elements alternate between the two inputs:
  <code>[A[0], B[0], A[1], B[1], A[2], B[2], ...]</code>
</p>

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 400 180" width="400" height="180"
     style="display:block; margin:20px auto;" font-family="monospace" font-size="11">
  <defs>
    <marker id="arrBlue" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 0 L 10 5 L 0 10 z" fill="#4477bb"/>
    </marker>
    <marker id="arrGreen" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 0 L 10 5 L 0 10 z" fill="#44aa66"/>
    </marker>
  </defs>
  <rect width="400" height="180" rx="8" fill="#222"/>

  <!-- Label A -->
  <text x="60" y="20" fill="#ccc" font-size="13" text-anchor="middle" font-weight="bold">A</text>
  <!-- Array A cells -->
  <rect x="10"  y="26" width="40" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="30"  y="45" text-anchor="middle" fill="#ccc">a&#x2080;</text>
  <rect x="54"  y="26" width="40" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="74"  y="45" text-anchor="middle" fill="#ccc">a&#x2081;</text>
  <rect x="98"  y="26" width="40" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="118" y="45" text-anchor="middle" fill="#ccc">a&#x2082;</text>
  <rect x="142" y="26" width="40" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="162" y="45" text-anchor="middle" fill="#ccc">a&#x2083;</text>

  <!-- Label B -->
  <text x="310" y="20" fill="#ccc" font-size="13" text-anchor="middle" font-weight="bold">B</text>
  <!-- Array B cells -->
  <rect x="220" y="26" width="40" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="240" y="45" text-anchor="middle" fill="#ccc">b&#x2080;</text>
  <rect x="264" y="26" width="40" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="284" y="45" text-anchor="middle" fill="#ccc">b&#x2081;</text>
  <rect x="308" y="26" width="40" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="328" y="45" text-anchor="middle" fill="#ccc">b&#x2082;</text>
  <rect x="352" y="26" width="40" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="372" y="45" text-anchor="middle" fill="#ccc">b&#x2083;</text>

  <!-- Label output -->
  <text x="200" y="108" fill="#ccc" font-size="13" text-anchor="middle" font-weight="bold">output</text>
  <!-- Output array cells (alternating blue/green) -->
  <rect x="10"  y="114" width="42" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="31"  y="133" text-anchor="middle" fill="#ccc">a&#x2080;</text>
  <rect x="56"  y="114" width="42" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="77"  y="133" text-anchor="middle" fill="#ccc">b&#x2080;</text>
  <rect x="102" y="114" width="42" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="123" y="133" text-anchor="middle" fill="#ccc">a&#x2081;</text>
  <rect x="148" y="114" width="42" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="169" y="133" text-anchor="middle" fill="#ccc">b&#x2081;</text>
  <rect x="194" y="114" width="42" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="215" y="133" text-anchor="middle" fill="#ccc">a&#x2082;</text>
  <rect x="240" y="114" width="42" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="261" y="133" text-anchor="middle" fill="#ccc">b&#x2082;</text>
  <rect x="286" y="114" width="42" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="307" y="133" text-anchor="middle" fill="#ccc">a&#x2083;</text>
  <rect x="332" y="114" width="42" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="353" y="133" text-anchor="middle" fill="#ccc">b&#x2083;</text>

  <!-- Curved arrows from A to output (dashed, blue) -->
  <path d="M30,54 C30,80 31,90 31,114" stroke="#4477bb" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrBlue)"/>
  <path d="M74,54 C74,78 123,90 123,114" stroke="#4477bb" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrBlue)"/>
  <path d="M118,54 C118,78 215,90 215,114" stroke="#4477bb" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrBlue)"/>
  <path d="M162,54 C162,78 307,90 307,114" stroke="#4477bb" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrBlue)"/>

  <!-- Curved arrows from B to output (dashed, green) -->
  <path d="M240,54 C240,78 77,90 77,114" stroke="#44aa66" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrGreen)"/>
  <path d="M284,54 C284,78 169,90 169,114" stroke="#44aa66" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrGreen)"/>
  <path d="M328,54 C328,78 261,90 261,114" stroke="#44aa66" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrGreen)"/>
  <path d="M372,54 C372,78 353,90 353,114" stroke="#44aa66" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrGreen)"/>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>output</code> array</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:  A = [1.0, 2.0, 3.0], B = [4.0, 5.0, 6.0]
Output: [1.0, 4.0, 2.0, 5.0, 3.0, 6.0]
</pre>

<h2>Example 2:</h2>
<pre>
Input:  A = [10.0, 20.0], B = [30.0, 40.0]
Output: [10.0, 30.0, 20.0, 40.0]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>N</code> &le; 50,000,000</li>

  <li>Performance is measured with <code>N</code> = 25,000,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__global__ void interleave_kernel(const float* A, const float* B, float* output, int N) {}

// A, B, output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const float* A, const float* B, float* output, int N) {
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    interleave_kernel<<<blocksPerGrid, threadsPerBlock>>>(A, B, output, N);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# A, B, output are tensors on the GPU
@cute.jit
def solve(A: cute.Tensor, B: cute.Tensor, output: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# A, B are tensors on the GPU
@jax.jit
def solve(A: jax.Array, B: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def interleave_kernel(
    A: UnsafePointer[Float32, MutExternalOrigin],
    B: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
):
    pass


# A, B, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    A: UnsafePointer[Float32, MutExternalOrigin],
    B: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
) raises:
    var BLOCK_SIZE: Int32 = 256
    var ctx = DeviceContext()
    var num_blocks = ceildiv(N, BLOCK_SIZE)

    var _kernel = ctx.compile_function[interleave_kernel, interleave_kernel]()
    ctx.enqueue_function(_kernel, A, B, output, N, grid_dim=num_blocks, block_dim=BLOCK_SIZE)

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# A, B, output are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, output: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


@triton.jit
def interleave_kernel(A_ptr, B_ptr, output_ptr, N, BLOCK_SIZE: tl.constexpr):
    pass


# A, B, output are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, output: torch.Tensor, N: int):
    BLOCK_SIZE = 256

    def grid(meta):
        return (triton.cdiv(N, meta["BLOCK_SIZE"]),)

    interleave_kernel[grid](A, B, output, N, BLOCK_SIZE=BLOCK_SIZE)


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/easy/63_interleave/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
